In [11]:
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_message_histories import ChatMessageHistory
import time
import os

# Set your Gemini Key


GOOGLE_API_KEY=os.environ.get("GOOGLE_API_KEY")

def start_rua_hybrid_master():
    # --- Load Models ---
    print("👂 Powering up RUA's Ear (Faster-Whisper)...")
    stt_model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=4)
    
    print("🏠 Initializing Local Brain (Llama 3.2)...")
    local_llm = ChatOllama(model="llama3.2", temperature=0.3)
    
    print("☁️  Initializing Cloud Brain (Gemini 2.5)...")
    cloud_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3,google_api_key=GOOGLE_API_KEY)
    
    memory = ChatMessageHistory()
    recognizer = sr.Recognizer()
    recognizer.pause_threshold = 0.8
    recognizer.non_speaking_duration = 0.5

    print("\n⚡ RUA Hybrid v5: Llama + Gemini + Word-by-Word ON.")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            recognizer.adjust_for_ambient_noise(source, duration=1.0)
            
            while True:
                print("\n🟢 Listening...")
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=10)
                
                # --- 1. STT STAGE (Word-by-Word Print) ---
                stt_start = time.time()
                raw_data = audio.get_raw_data()
                audio_np = np.frombuffer(raw_data, dtype=np.int16).astype(np.float32) / 32768.0
                
                segments, info = stt_model.transcribe(audio_np, beam_size=1, word_timestamps=True)
                
                detected_lang = info.language
                print(f"👤 You ({detected_lang.upper()}): ", end="", flush=True)
                
                full_user_text = ""
                for segment in segments:
                    for word in segment.words:
                        print(word.word, end="", flush=True)
                        full_user_text += word.word
                
                stt_latency = time.time() - stt_start

                if full_user_text.strip():
                    # --- 2. HYBRID ROUTING LOGIC ---
                    # Trigger Cloud for complex tasks or explicit requests
                    complex_keywords = ['research', 'explain', 'code', 'write', 'summarize', 'plan']
                    is_complex = any(k in full_user_text.lower() for k in complex_keywords)
                    
                    if is_complex:
                        selected_llm = cloud_llm
                        brain_label = "GEMINI (Cloud)"
                    else:
                        selected_llm = local_llm
                        brain_label = "LLAMA (Local)"

                    # --- 3. DYNAMIC SYSTEM PROMPT ---
                    if detected_lang == "en":
                        lang_instruction = "Respond ONLY in English. NO Hindi."
                    else:
                        lang_instruction = "Respond in natural Hinglish (Hindi + English mix)."
                    
                    system_prompt = f"""Your name is RUA. {lang_instruction}
                    1. Be witty and concise (under 30 words).
                    2. The user is Yanil. If they ask for their name, it is Yanil.
                    3. Use history for context. No markdown/bold."""

                    # --- 4. LLM STAGE ---
                    llm_start = time.time()
                    print(f"\n🤖 RUA [{brain_label}]: ", end="", flush=True)
                    
                    messages = [{"role": "system", "content": system_prompt}]
                    # Feed last 6 messages from memory
                    for msg in memory.messages[-6:]:
                        role = "user" if msg.type == "human" else "assistant"
                        messages.append({"role": role, "content": msg.content})
                    messages.append({"role": "user", "content": full_user_text})
                    
                    full_response = ""
                    # Stream the response chunk by chunk
                    for chunk in selected_llm.stream(messages):
                        # Handle content extraction for different LLM types
                        content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                        print(content, end="", flush=True)
                        full_response += content
                    
                    llm_latency = time.time() - llm_start
                    
                    # Update History
                    memory.add_user_message(full_user_text)
                    memory.add_ai_message(full_response)
                    
                    print(f"\n\n[METRICS] STT: {stt_latency:.2f}s | LLM ({brain_label}): {llm_latency:.2f}s")
                    print("-" * 30)
                else:
                    print("\r⚪ Silence.")

    except KeyboardInterrupt:
        print("\n🛑 RUA shutting down.")

if __name__ == "__main__":
    start_rua_hybrid_master()

👂 Powering up RUA's Ear (Faster-Whisper)...
🏠 Initializing Local Brain (Llama 3.2)...
☁️  Initializing Cloud Brain (Gemini 2.5)...

⚡ RUA Hybrid v5: Llama + Gemini + Word-by-Word ON.

🟢 Listening...
👤 You (EN):  Explain the engine.
🤖 RUA [GEMINI (Cloud)]: The engine? It's the heart of motion, Yanil, turning fuel into vroom-vroom. Without it, you're just pushing.

[METRICS] STT: 5.48s | LLM (GEMINI (Cloud)): 4.68s
------------------------------

🟢 Listening...
👤 You (NN): 
🛑 RUA shutting down.


In [ ]:
# hybrid_llm_brain.py
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_message_histories import ChatMessageHistory
import os
from memory_management import rua_brain, identity_router

def start_rua_hybrid_master():
    # --- Init ---
    stt_model = WhisperModel("small", device="cpu", compute_type="int8")
    local_llm = ChatOllama(model="llama3.2", temperature=0.3)
    cloud_llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", google_api_key=os.environ.get("GOOGLE_API_KEY"))
    session_history = ChatMessageHistory()
    recognizer = sr.Recognizer()

    print("⚡ RUA v7: Multi-User Voice Profiles Enabled.")
    print("👋 RUA: Who am I speaking with today?")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            recognizer.adjust_for_ambient_noise(source)
            while True:
                print(f"\n🟢 Listening ({rua_brain.active_user})...")
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=10)
                
                # --- STT ---
                audio_np = np.frombuffer(audio.get_raw_data(), dtype=np.int16).astype(np.float32) / 32768.0
                segments, _ = stt_model.transcribe(audio_np, beam_size=1)
                full_user_text = "".join([s.text for s in segments]).strip()
                
                if full_user_text:
                    print(f"👤 {rua_brain.active_user}: {full_user_text}")

                    # --- 1. Identity Check ---
                    identity_change = identity_router(full_user_text)
                    
                    # --- 2. Selective History ---
                    is_follow_up = any(w in full_user_text.lower() for w in ['it', 'they', 'him', 'her', 'why'])
                    
                    # --- 3. Dynamic Multi-User Prompt ---
                    system_prompt = f"""Your name is RUA. 
                    ACTIVE PROFILE: {rua_brain.get_brain_dump()}
                    1. Be witty and personalized to this specific user.
                    2. Under 30 words. No markdown."""

                    messages = [{"role": "system", "content": system_prompt}]
                    if is_follow_up:
                        for msg in session_history.messages[-4:]:
                            messages.append({"role": "user" if msg.type == "human" else "assistant", "content": msg.content})
                    messages.append({"role": "user", "content": full_user_text})

                    # --- 4. Brain Routing ---
                    llm = cloud_llm if "research" in full_user_text.lower() else local_llm
                    print(f"🤖 RUA: ", end="", flush=True)
                    full_response = ""
                    for chunk in llm.stream(messages):
                        content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                        print(content, end="", flush=True)
                        full_response += content

                    session_history.add_user_message(full_user_text)
                    session_history.add_ai_message(full_response)
                else:
                    print("\r⚪ Silence.")

    except KeyboardInterrupt:
        print(f"\n🛑 Shutting down {rua_brain.active_user}'s session...")
        # Summarize for the specific user
        summary = f"Session with {rua_brain.active_user} focused on {full_user_text[:50]}"
        rua_brain.memory["users"][rua_brain.active_user]["episodic_memory"].append({"date": "now", "summary": summary})
        
        # Consolidate for the active user
        print(rua_brain.consolidate(cloud_llm))

if __name__ == "__main__":
    start_rua_hybrid_master()